# Fold 1 detector diagnostics and redevelopment

## TL;DR

The original 2021 Fold 1 detector remains failed. This notebook reproduces the 2018–2021 diagnostic artifacts, verifies family-week equal-volume comparisons, and reviews the candidate recommended for an untouched 2022 test. It does **not** execute Fold 2 or claim that the detector works.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'ROLE_CHANGE_VALIDATION_PROTOCOL.md').exists():
    candidates = [p for p in [ROOT, *ROOT.parents] if (p / 'ROLE_CHANGE_VALIDATION_PROTOCOL.md').exists()]
    if not candidates:
        raise RuntimeError('Repository root not found')
    ROOT = candidates[0]
OUT = ROOT / 'outputs' / 'role_validation' / 'fold_1_diagnostics'
ALLOWED = {2018, 2019, 2020, 2021}
PRIMARY = 'PRIMARY_CONFIRMED_EXCLUDED'
RECOMMENDED = 'fold2_candidate_v1_symmetric_deltas'

def read(name):
    frame = pd.read_csv(OUT / name, low_memory=False)
    if 'season' in frame:
        observed = set(pd.to_numeric(frame['season'], errors='coerce').dropna().astype(int))
        assert observed <= ALLOWED, (name, observed)
    return frame

manifest = json.loads((OUT / 'run_manifest.json').read_text())
assert manifest['fold_2_executed'] is False
assert manifest['post_2021_results_used'] is False
assert manifest['release_gates_changed'] is False
manifest

{'stage': 'final',
 'checkpoint_commit': '00d6085a55c60147e0ace46c847460ef5708e968',
 'allowed_seasons': [2018, 2019, 2020, 2021],
 'development_seasons': [2018, 2019, 2020],
 'test_season': 2021,
 'fold_2_executed': False,
 'post_2021_results_used': False,
 'release_gates_changed': False,
 'canonical_rows_2018_2021': 28199,
 'original_full_family_alerts_2021': 717,
 'original_deduplicated_feed_alerts_2021': 489,
 'explicit_partial_family_rows': 96,
 'suspected_partial_family_rows': 298,
 'screen_candidates_attempted': 54,
 'screen_candidates_valid': 53,
 'screen_candidates_integrity_failures': 1,
 'serious_candidates': ['S1_corrected_minimal',
  'S2_balanced_directional_no_cooldown',
  'S2_balanced_directional',
  'S2_symmetric_deltas',
  'S3_balanced_season_recent',
  'S4_high_specificity'],
 'recommended_config': 'config/role_change_fold2_candidate.yaml',
 'all_serious_equal_volume': True,
 'expected_methods_present': True}

## Context & Methods

The checkpoint detector is diagnosed at family-alert grain and at the deduplicated player-week-team feed grain. Revised features use a same-season, disjoint baseline that ends before the confirmation window. Comparators are selected to exactly the full-detector count within family-season-week. Precision uses the locked 2,000-draw bootstrap; improvement uses a season-week cluster bootstrap.

### Key Assumptions

- Only 2018–2020 development data and revised 2021 evidence are loaded.
- 2021 is no longer treated as untouched after redevelopment.
- Suspected partial games remain in the primary analysis.
- A confirmed partial requires explicit PBP evidence and a valid pre-next-team-game window.
- Locked release gates are diagnostic only and are not modified.

## Data

In [2]:
audit = read('canonical_redevelopment_audit_2018_2021.csv')
missing = read('canonical_redevelopment_missingness_2018_2021.csv')
partial_source = read('partial_game_source_coverage.csv')
partial_counts = read('partial_game_status_counts.csv')
assert audit['duplicate_key_rows'].sum() == 0
assert missing['null_rows'].sum() == 0
assert partial_source['trigger_timestamp_missing_team_games'].eq(0).all()
display(audit)
display(partial_source)
display(partial_counts)

,season,canonical_rows,unique_players,duplicate_key_rows,duplicate_key_rate,required_field_null_cells,quality_fail_rows,quality_pass_rate,qualifying_rate,confirmed_partial_family_rows,suspected_partial_family_rows
0,2018,6777,526,0,0.0,0,225,0.966799,0.966799,16,77
1,2019,6816,529,0,0.0,0,191,0.971978,0.971978,23,67
2,2020,7134,557,0,0.0,0,0,1.000000,1.000000,23,75
3,2021,7472,587,0,0.0,0,0,1.000000,1.000000,34,79


,parsed_injury_mentions,resolved_injury_mentions,unresolved_injury_mentions,resolution_rate,ambiguous_roster_universe_keys_excluded,participation_team_games,participation_coverage_below_099,canonical_rows,confirmed_partial_rows,suspected_partial_rows,statistical_corroboration_rows_pre_promotion,suspected_corroborated_status_rows,canonical_team_games,trigger_timestamp_missing_team_games,next_boundary_missing_team_games
0,3187,2929,258,0.919046,19033,2080,2,28199,96,298,262,184,2080,0,128


,season,partial_game_status,canonical_family_rows,distinct_player_games
0,2018,confirmed,16,11
1,2018,none,6684,5236
2,2018,suspected_corroborated,44,38
3,2018,suspected_statistical,33,25
4,2019,confirmed,23,16
5,2019,none,6726,5239
6,2019,suspected_corroborated,45,36
7,2019,suspected_statistical,22,16
8,2020,confirmed,23,19
9,2020,none,7036,5483


## Results

### Original Fold 1 volume, overlap, and repeats

In [3]:
weekly_original = read('original_weekly_family_vs_deduplicated_volume_2021.csv')
rb_overlap = read('original_rb_family_overlap_2021.csv')
repeats = read('original_repeat_alerts_2021.csv')
assert weekly_original['family_alert_rows'].sum() == 717
assert weekly_original['deduplicated_feed_alerts'].sum() == 489
display(weekly_original)
display(rb_overlap)
display(repeats)

,season,week,family_alert_rows,deduplicated_feed_alerts,duplicate_family_rows_removed
0,2021,1,40,27,13
1,2021,2,51,33,18
2,2021,3,40,30,10
3,2021,4,42,30,12
4,2021,5,36,22,14
5,2021,6,27,18,9
6,2021,7,35,22,13
7,2021,8,31,22,9
8,2021,9,33,24,9
9,2021,10,29,19,10


,carry_alerts,opportunity_alerts,overlap_alerts,union_alerts,carry_overlap_rate,opportunity_overlap_rate,jaccard_overlap,direction_conflicts
0,273,324,228,369,0.835165,0.703704,0.617886,0


,grain,alerts,repeat_alerts,repeat_rate,players_with_repeat
0,deduplicated_player_week,489,151,0.308793,70
1,family_player_week,717,219,0.305439,69
2,family:rb_carry_share,273,90,0.329670,43
3,family:rb_opportunity_share,324,113,0.348765,50
4,family:te_target_share,35,6,0.171429,6
5,family:wr_target_share,85,10,0.117647,9


### Original method comparison and requested breakdowns

In [4]:
methods = read('original_four_method_comparison_2021.csv')
breakdowns = read('original_requested_breakdowns_2021.csv')
display(methods[methods['grain'].eq('all_family_rows')])
for dimension in ['role_family', 'direction', 'baseline_sample_bin', 'raw_player_opportunities',
                  'team_opportunity_denominator', 'absolute_detected_change', 'partial_game_status']:
    display(breakdowns[breakdowns['dimension'].eq(dimension)])

,grain,method,role_family,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks,precision_ci_low,precision_ci_high
16,all_family_rows,full_propwar,ALL,717,562,0.783821,308,0.548043,626,203,0.324281,0.586970,0.577473,181,18,NaN,NaN
17,all_family_rows,naive_spike,ALL,717,565,0.788006,267,0.472566,628,228,0.363057,0.453152,0.437065,194,18,NaN,NaN
18,all_family_rows,normal_game_trend,ALL,717,562,0.783821,304,0.540925,628,209,0.332803,0.568417,0.568231,181,18,NaN,NaN
19,all_family_rows,two_week_raw,ALL,717,566,0.789400,302,0.533569,625,210,0.336000,0.561825,0.600041,180,18,NaN,NaN


,candidate_name,partial_policy,period,method,dimension,segment,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
0,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,role_family,rb_carry_share,273,222,0.813187,127,0.572072,238,67,0.281513,0.657328,0.612822,93,18
1,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,role_family,rb_opportunity_share,324,256,0.790123,150,0.585938,282,85,0.301418,0.642635,0.616964,103,18
2,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,role_family,te_target_share,35,25,0.714286,5,0.200000,32,16,0.500000,0.165951,0.153888,22,18
3,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,role_family,wr_target_share,85,59,0.694118,26,0.440678,74,35,0.472973,0.400202,0.452598,53,18


,candidate_name,partial_policy,period,method,dimension,segment,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
4,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,direction,decrease,342,262,0.766082,158,0.603053,286,74,0.258741,0.744618,0.641383,119,18
5,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,direction,increase,375,300,0.800000,150,0.500000,340,129,0.379412,0.498089,0.521657,135,18


,candidate_name,partial_policy,period,method,dimension,segment,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
27,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,baseline_sample_bin,3,1,1,1.000000,1,1.000000,1,0,0.000000,1.738206,1.738206,1,1
28,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,baseline_sample_bin,4,681,536,0.787078,302,0.563433,593,187,0.315346,0.604601,0.595064,159,18
29,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,baseline_sample_bin,5+,35,25,0.714286,5,0.200000,32,16,0.500000,0.165951,0.153888,22,18


,candidate_name,partial_policy,period,method,dimension,segment,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
30,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,raw_player_opportunities,0-2,162,124,0.765432,72,0.580645,141,35,0.248227,0.646086,0.492614,77,18
31,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,raw_player_opportunities,10-14,130,103,0.792308,51,0.495146,112,42,0.375000,0.483565,0.558015,74,18
32,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,raw_player_opportunities,15+,169,141,0.834320,76,0.539007,155,52,0.335484,0.590686,0.548637,60,18
33,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,raw_player_opportunities,3-5,117,85,0.726496,54,0.635294,98,32,0.326531,0.794323,0.696908,69,18
34,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,raw_player_opportunities,6-9,139,109,0.784173,55,0.504587,120,42,0.350000,0.503305,0.636558,86,18


,candidate_name,partial_policy,period,method,dimension,segment,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
35,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,team_opportunity_denominator,0-15,87,71,0.816092,38,0.535211,84,26,0.309524,0.527269,0.510070,51,17
36,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,team_opportunity_denominator,16-20,136,117,0.860294,55,0.470085,125,43,0.344000,0.408426,0.474630,76,18
37,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,team_opportunity_denominator,21-25,212,154,0.726415,82,0.532468,178,62,0.348315,0.558146,0.523313,105,18
38,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,team_opportunity_denominator,26-30,150,113,0.753333,74,0.654867,126,35,0.277778,0.677460,0.723206,86,18
39,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,team_opportunity_denominator,31-35,77,67,0.870130,37,0.552239,70,21,0.300000,0.590686,0.630217,57,18
40,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,team_opportunity_denominator,36+,55,40,0.727273,22,0.550000,43,16,0.372093,0.834332,0.706402,42,16


,candidate_name,partial_policy,period,method,dimension,segment,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
41,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,absolute_detected_change,0.10-0.149,93,63,0.677419,28,0.444444,82,38,0.463415,0.400202,0.409453,67,18
42,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,absolute_detected_change,0.15-0.199,251,206,0.820717,97,0.470874,223,88,0.394619,0.404738,0.472384,110,18
43,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,absolute_detected_change,0.20-0.249,154,114,0.740260,67,0.587719,133,39,0.293233,0.654230,0.688868,75,18
44,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,absolute_detected_change,0.25+,219,179,0.817352,116,0.648045,188,38,0.202128,0.806306,0.686603,59,18


,candidate_name,partial_policy,period,method,dimension,segment,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
45,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,partial_game_status,confirmed,11,5,0.454545,3,0.600000,7,2,0.285714,1.416346,1.183653,7,6
46,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,partial_game_status,none,685,541,0.789781,297,0.548983,603,199,0.330017,0.584022,0.567197,176,18
47,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,partial_game_status,suspected_corroborated,9,8,0.888889,1,0.125000,8,2,0.250000,0.223004,0.058516,6,6
48,original_checkpoint_00d6085,CHECKPOINT_NO_RELIABLE_PARTIAL_EXCLUSION,fold_1_2021,full_propwar,partial_game_status,suspected_statistical,12,8,0.666667,7,0.875000,8,0,0.000000,1.349767,1.412460,6,6


### Safeguard ablations and false-positive reasons

In [5]:
ablation = read('legacy_safeguard_ablation_value.csv')
manual_path = OUT / 'original_false_positive_manual_adjudication_2021.csv'
false_positives = pd.read_csv(manual_path, low_memory=False) if manual_path.exists() else read('original_false_positive_case_review_2021.csv')
assert ablation.groupby(['ablation', 'ablation_mode'])['role_family'].nunique().eq(4).all()
display(ablation[ablation['ablation_mode'].eq('operational')])
reason_column = 'manual_primary_reason_code' if 'manual_primary_reason_code' in false_positives else 'primary_reason_code'
display(false_positives.groupby(reason_column).size().rename('cases').sort_values(ascending=False))

,ablation,ablated_safeguard,ablation_mode,role_family,alert_delta,precision_delta,reversion_rate_delta,median_retention_delta,full_alerts,overlap_with_original,added_vs_original,removed_vs_original,identical_membership,fixed_volume_backfill_rows,no_measurable_value
4,min_one_baseline_game,minimum_baseline_sample,operational,rb_carry_share,13,0.006810,-0.002625,0.026524,746,717,29,0,False,0,False
5,min_one_baseline_game,minimum_baseline_sample,operational,rb_opportunity_share,16,0.012035,-0.013043,0.058220,746,717,29,0,False,0,False
6,min_one_baseline_game,minimum_baseline_sample,operational,te_target_share,0,0.000000,0.000000,0.000000,746,717,29,0,False,0,False
7,min_one_baseline_game,minimum_baseline_sample,operational,wr_target_share,0,0.000000,0.000000,0.000000,746,717,29,0,False,0,False
12,no_concentration_penalty,concentration_penalty,operational,rb_carry_share,0,0.000000,0.000000,0.000000,717,717,0,0,True,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,no_two_game_persistence,two_game_persistence,operational,wr_target_share,257,-0.113472,-0.010432,-0.103702,1411,619,792,98,False,0,False
140,original_full_detector,none,operational,rb_carry_share,0,0.000000,0.000000,0.000000,717,717,0,0,True,0,True
141,original_full_detector,none,operational,rb_opportunity_share,0,0.000000,0.000000,0.000000,717,717,0,0,True,0,True
142,original_full_detector,none,operational,te_target_share,0,0.000000,0.000000,0.000000,717,717,0,0,True,0,True


manual_primary_reason_code
ROLE_REVERSION_NO_OBSERVED_DATA_ISSUE    84
BASELINE_SMALL_OR_UNSTABLE               53
LOW_TEAM_DENOMINATOR_NOISE               52
MARGINAL_CHANGE_NEAR_THRESHOLD           35
LOW_PLAYER_OPPORTUNITY_NOISE              8
SUSPECTED_FOCAL_PARTIAL                   8
NORMAL_CONTEXT_SENSITIVE                  7
SUSPECTED_TEAMMATE_EXIT_CONTEXT           5
CONFIRMED_FOCAL_PARTIAL                   2
Name: cases, dtype: int64

### Candidate screening and equal-volume integrity

In [6]:
screens = read('candidate_axis_screen_equal_volume.csv')
serious_equal = read('serious_candidate_equal_volume.csv')
sensitivity_equal = read('recommended_candidate_partial_sensitivity_equal_volume.csv')
assert screens['integrity_pass'].fillna(False).sum() == 53
assert (~screens['integrity_pass'].fillna(False)).sum() == 1
for check in [serious_equal, sensitivity_equal]:
    assert check['equal_volume'].all()
    assert check['observed_method_count'].eq(4).all()
display(screens[~screens['integrity_pass'].fillna(False)])

,candidate_name,partial_policy,family_weeks,equal_volume_all,expected_methods_all,integrity_pass,integrity_error,screening_axis,screening_level
5,screen_consecutive_confirmation_one_game_none,PRIMARY_CONFIRMED_EXCLUDED,0,False,False,False,Equal-volume selection impossible for screen_c...,consecutive_confirmation,one_game_none


### Original versus revised and recommended family results

In [7]:
original_vs_revised = read('original_vs_recommended_fold1_2021.csv')
family = read('recommended_candidate_partial_sensitivity_comparisons.csv')
primary = family[family['partial_policy'].eq(PRIMARY)]
display(original_vs_revised)
display(primary)

,role_family,original_full_alerts,original_full_evaluable_alerts,original_full_precision,original_naive_precision,original_precision_improvement,original_relative_precision_improvement,original_precision_improvement_ci_low,original_precision_improvement_ci_high,original_full_reversion_rate,...,revised_reversion_improvement,revised_full_median_retention,revised_naive_median_retention,delta_full_alerts,delta_full_evaluable_alerts,delta_full_precision,delta_precision_improvement,delta_full_reversion_rate,delta_reversion_improvement,delta_full_median_retention
0,rb_carry_share,273,222,0.572072,0.493213,0.078859,0.159889,0.021341,0.140606,0.281513,...,0.110372,0.761506,0.404684,-217,-179,0.172114,0.177522,-0.094013,0.046052,0.104178
1,rb_opportunity_share,324,256,0.585938,0.517510,0.068428,0.132225,0.001807,0.143318,0.301418,...,0.111111,0.684435,0.627807,-247,-201,0.086790,0.077027,-0.110942,0.089835,0.041801
2,te_target_share,35,25,0.200000,0.269231,-0.069231,-0.257143,-0.204585,0.066692,0.500000,...,0.000000,-0.141319,-0.118992,-31,-23,0.300000,0.569231,0.000000,0.045455,-0.307270
3,wr_target_share,85,59,0.440678,0.295082,0.145596,0.493409,0.015140,0.272534,0.472973,...,0.318182,0.551380,0.082288,-59,-43,0.121822,0.240433,-0.109337,0.256908,0.151178


,candidate_name,partial_policy,period,role_family,full_alerts,full_evaluable_alerts,full_precision,naive_precision,precision_improvement,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high
8,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,rb_carry_share,157,122,0.663934,0.444444,0.219490,0.211679,0.320896,0.109217,0.723026,0.439142,0.493852,0.125023,0.317760
9,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,rb_opportunity_share,219,167,0.622754,0.528662,0.094092,0.242268,0.297872,0.055604,0.704132,0.535112,0.177981,-0.004271,0.184442
10,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,te_target_share,4,4,0.500000,0.000000,0.500000,0.250000,0.500000,0.250000,0.491190,0.243863,NaN,0.000000,1.000000
11,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,wr_target_share,78,61,0.475410,0.225806,0.249603,0.239437,0.514706,0.275269,0.463288,0.276898,1.105386,0.086172,0.422425
12,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,fold_1_2021,rb_carry_share,56,43,0.744186,0.487805,0.256381,0.187500,0.297872,0.110372,0.761506,0.404684,0.525581,0.006882,0.483918
13,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,fold_1_2021,rb_opportunity_share,77,55,0.672727,0.527273,0.145455,0.190476,0.301587,0.111111,0.684435,0.627807,0.275862,-0.021458,0.277018
14,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,fold_1_2021,te_target_share,4,2,0.500000,0.000000,0.500000,0.500000,0.500000,0.000000,-0.141319,-0.118992,NaN,0.000000,1.000000
15,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,fold_1_2021,wr_target_share,26,16,0.562500,0.176471,0.386029,0.363636,0.681818,0.318182,0.551380,0.082288,2.187500,0.176471,0.647059


### Direction, week blocks, feed volume, and sensitivities

In [8]:
direction = read('recommended_candidate_partial_sensitivity_direction.csv')
blocks = read('recommended_candidate_partial_sensitivity_block_comparisons.csv')
weekly = read('recommended_candidate_partial_sensitivity_weekly_2021.csv')
feed = read('recommended_candidate_partial_sensitivity_feed_summary_2021.csv')
thresholds = read('recommended_candidate_persistence_threshold_sensitivity.csv')
display(direction[direction['partial_policy'].eq(PRIMARY)])
display(blocks[blocks['partial_policy'].eq(PRIMARY)])
display(weekly[weekly['partial_policy'].eq(PRIMARY)])
display(feed)
display(thresholds)

,candidate_name,partial_policy,period,role_family,direction,alerts,evaluable_alerts,evaluable_rate,persistent_alerts,precision,reversion_evaluable_alerts,immediate_reversions,reversion_rate,median_retention,mean_retention,unique_players,active_weeks
16,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,rb_carry_share,decrease,66,50,0.757576,33,0.660000,57,13,0.228070,0.777014,0.731372,44,29
17,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,rb_carry_share,increase,91,72,0.791209,48,0.666667,80,16,0.200000,0.676684,0.661647,66,34
18,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,rb_opportunity_share,decrease,108,84,0.777778,53,0.630952,97,26,0.268041,0.707706,0.707990,64,30
19,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,rb_opportunity_share,increase,111,83,0.747748,51,0.614458,97,21,0.216495,0.704132,0.674627,70,34
20,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,te_target_share,decrease,1,1,1.000000,0,0.000000,1,0,0.000000,-0.162208,-0.162208,1,1
21,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,te_target_share,increase,3,3,1.000000,2,0.666667,3,1,0.333333,0.678528,0.635707,3,3
22,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,wr_target_share,decrease,34,28,0.823529,12,0.428571,32,8,0.250000,0.447660,0.494565,29,23
23,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,development_2018_2020,wr_target_share,increase,44,33,0.750000,17,0.515152,39,9,0.230769,0.510582,0.524826,38,24
24,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,fold_1_2021,rb_carry_share,decrease,24,19,0.791667,15,0.789474,19,3,0.157895,0.864068,0.805641,20,10
25,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,fold_1_2021,rb_carry_share,increase,32,24,0.750000,17,0.708333,29,6,0.206897,0.738174,0.766831,28,13


,candidate_name,partial_policy,week_block,role_family,full_alerts,full_evaluable_alerts,full_precision,naive_precision,precision_improvement,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention
11,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_13_18,rb_carry_share,32,19,0.684211,0.529412,0.154799,0.333333,0.260870,-0.072464,0.761506,0.663462
12,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_13_18,rb_opportunity_share,46,25,0.760000,0.500000,0.260000,0.218750,0.343750,0.125000,0.778376,0.475013
13,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_13_18,te_target_share,3,1,0.000000,0.000000,0.000000,1.000000,0.000000,-1.000000,-0.813236,-0.091852
14,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_13_18,wr_target_share,15,6,0.666667,0.285714,0.380952,0.454545,0.818182,0.363636,0.726959,0.082288
15,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_1_6,rb_carry_share,3,3,0.333333,1.000000,-0.666667,0.000000,0.000000,0.000000,0.000806,0.674820
16,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_1_6,rb_opportunity_share,4,4,0.500000,1.000000,-0.500000,0.250000,0.000000,-0.250000,0.307868,0.876112
17,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_1_6,wr_target_share,2,2,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.749529,0.424881
18,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_7_12,rb_carry_share,21,21,0.857143,0.380952,0.476190,0.047619,0.380952,0.333333,0.864068,0.306919
19,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_7_12,rb_opportunity_share,27,26,0.615385,0.481481,0.133903,0.148148,0.296296,0.148148,0.681748,0.497092
20,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,weeks_7_12,te_target_share,1,1,1.000000,0.000000,1.000000,0.000000,1.000000,1.000000,0.530598,-0.146132


,candidate_name,partial_policy,season,week,family_alert_rows,deduplicated_feed_alerts,duplicate_family_rows_removed
0,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,1,0,0,0
1,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,2,0,0,0
2,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,3,0,0,0
3,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,4,0,0,0
4,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,5,0,0,0
5,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,6,9,6,3
6,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,7,10,7,3
7,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,8,4,3,1
8,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,9,13,10,3
9,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,2021,10,8,6,2


,candidate_name,partial_policy,family_alert_rows,deduplicated_feed_alerts,duplicate_family_rows_removed,median_all_18_weeks,median_active_weeks,mean_all_18_weeks,p90_all_18_weeks,max_week,zero_alert_weeks,weeks_above_15,weeks_above_20,within_5_15_median_target
0,fold2_candidate_v1_symmetric_deltas,PRIMARY_CONFIRMED_EXCLUDED,163,122,41,7.5,10.0,6.777778,11.9,17,5,1,0,True
1,fold2_candidate_v1_symmetric_deltas,ALL_INCLUDED,167,125,42,7.5,10.0,6.944444,12.2,17,5,1,0,True
2,fold2_candidate_v1_symmetric_deltas,STRICT_SUSPECTED_EXCLUDED,160,121,39,6.0,9.0,6.722222,13.3,15,5,0,0,True


,persistence_threshold,period,role_family,full_alerts,full_evaluable_alerts,full_precision,naive_precision,precision_improvement,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high
0,0.4,development_2018_2020,rb_carry_share,157,122,0.713115,0.529915,0.183200,0.211679,0.320896,0.109217,0.723026,0.439142,0.345717,0.075666,0.284734
1,0.4,development_2018_2020,rb_opportunity_share,219,167,0.694611,0.592357,0.102254,0.242268,0.297872,0.055604,0.704132,0.535112,0.172622,0.015258,0.191938
2,0.4,development_2018_2020,te_target_share,4,4,0.500000,0.333333,0.166667,0.250000,0.500000,0.250000,0.491190,0.243863,0.500000,-0.750000,1.000000
3,0.4,development_2018_2020,wr_target_share,78,61,0.573770,0.322581,0.251190,0.239437,0.514706,0.275269,0.463288,0.276898,0.778689,0.065376,0.425821
4,0.4,fold_1_2021,rb_carry_share,56,43,0.767442,0.512195,0.255247,0.187500,0.297872,0.110372,0.761506,0.404684,0.498339,-0.001776,0.489331
5,0.4,fold_1_2021,rb_opportunity_share,77,55,0.745455,0.563636,0.181818,0.190476,0.301587,0.111111,0.684435,0.627807,0.322581,0.019231,0.300000
6,0.4,fold_1_2021,te_target_share,4,2,0.500000,0.000000,0.500000,0.500000,0.500000,0.000000,-0.141319,-0.118992,NaN,0.000000,1.000000
7,0.4,fold_1_2021,wr_target_share,26,16,0.625000,0.235294,0.389706,0.363636,0.681818,0.318182,0.551380,0.082288,1.656250,0.235260,0.535714
8,0.5,development_2018_2020,rb_carry_share,157,122,0.663934,0.444444,0.219490,0.211679,0.320896,0.109217,0.723026,0.439142,0.493852,0.125023,0.317760
9,0.5,development_2018_2020,rb_opportunity_share,219,167,0.622754,0.528662,0.094092,0.242268,0.297872,0.055604,0.704132,0.535112,0.177981,-0.004271,0.184442


### Locked-gate diagnostic

In [9]:
gates = read('recommended_candidate_locked_gate_diagnostic_2021.csv')
assert gates['fold_2_executed'].eq(False).all()
assert gates['release_gates_changed'].eq(False).all()
assert gates['frozen_before_2021'].eq(False).all()
display(gates)

,role_family,release_status,point_gate_result,alerts,precision,naive_precision,precision_improvement,reversion_rate,reversion_improvement,median_retention,alerts_per_week,failed_checks,frozen_before_2021,fold_2_executed,release_gates_changed
0,rb_carry_share,DIAGNOSTIC_ONLY_2021_REUSED_FOR_REDEVELOPMENT,POINT_GATES_PASS,56,0.744186,0.487805,0.256381,0.187500,0.110372,0.761506,3.111111,NaN,False,False,False
1,rb_opportunity_share,DIAGNOSTIC_ONLY_2021_REUSED_FOR_REDEVELOPMENT,POINT_GATES_PASS,77,0.672727,0.527273,0.145455,0.190476,0.111111,0.684435,4.277778,NaN,False,False,False
2,te_target_share,DIAGNOSTIC_ONLY_2021_REUSED_FOR_REDEVELOPMENT,POINT_GATES_FAIL,4,0.500000,0.000000,0.500000,0.500000,0.000000,-0.141319,0.222222,"min_holdout_alerts, min_persistence_precision,...",False,False,False
3,wr_target_share,DIAGNOSTIC_ONLY_2021_REUSED_FOR_REDEVELOPMENT,POINT_GATES_FAIL,26,0.562500,0.176471,0.386029,0.363636,0.318182,0.551380,1.444444,"min_holdout_alerts, min_persistence_precision,...",False,False,False


## Takeaways

- RB-family duplication explains part, but not most, of the original volume failure.
- The original score weighting and concentration penalty have no selection effect.
- The recommended symmetric candidate meets the combined-feed median operating target and has the strongest evidence for RB carry; RB opportunity carries a development reversion caveat.
- WR and TE remain shadow-only because the evidence is sparse or unstable.
- Revised 2021 is development evidence. Fold 2 remains unexecuted and is the next untouched test.